# Notebook 5: ROI & Cost-Benefit Summary
Combines cost, scholarship, and professional outcome data into expected-value scenarios.
Monte Carlo simulation for uncertainty modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

costs  = pd.read_csv('../data/processed/costs.csv')
schols = pd.read_csv('../data/processed/scholarship_outcomes.csv')
pros   = pd.read_csv('../data/processed/pro_outcomes.csv')

In [ ]:
# --- 5A: Scenario definitions ---
# Career cost inputs (mid estimates)
career_cost = {
    1: costs[costs['tier']==1]['total_mid'].mean() * 13,
    2: costs[costs['tier']==2]['total_mid'].mean() * 9,
    3: costs[costs['tier']==3]['total_mid'].mean() * 7,
    4: 0,  # Pro academies are free
}

# Scholarship values (4-year awards)
d1_men_4yr   = schols[(schols['division']=='D1')&(schols['gender']=='Men')&(schols['year']==2024)]['avg_scholarship_value_usd'].values[0] * 4
d1_women_4yr = schols[(schols['division']=='D1')&(schols['gender']=='Women')&(schols['year']==2024)]['avg_scholarship_value_usd'].values[0] * 4
d2_men_4yr   = schols[(schols['division']=='D2')&(schols['gender']=='Men')&(schols['year']==2024)]['avg_scholarship_value_usd'].values[0] * 4

# Pro salary values (5-year career assumption, mid estimate)
mls_5yr  = pros[(pros['league']=='MLS')&(pros['year']==2026)]['salary_usd_mid'].values[0] * 5
nwsl_5yr = pros[(pros['league']=='NWSL')&(pros['year']==2024)]['salary_usd_mid'].values[0] * 5
euro_5yr = pros[pros['league']=='Bundesliga']['salary_usd_mid'].values[0] * 5

print('Career investment (mid est):')
for t, v in career_cost.items(): print(f'  Tier {t}: ${v:,.0f}')
print(f'\nScholarship returns (4yr): D1 Men=${d1_men_4yr:,.0f}  D1 Women=${d1_women_4yr:,.0f}')
print(f'Pro returns (5yr): MLS=${mls_5yr:,.0f}  NWSL=${nwsl_5yr:,.0f}  Bundesliga=${euro_5yr:,.0f}')

In [ ]:
# --- 5B: Expected value table ---
# Probabilities (realistic estimates based on player funnel from notebook 4)
prob = {
    'D1 schol from Tier 3': 0.08,     # ~8% of ECNL players get D1 scholarship
    'Any schol from Tier 3': 0.35,    # ~35% of ECNL players get some scholarship
    'Pro contract from Tier 3': 0.02, # ~2% from elite private clubs
    'Pro contract from Tier 4': 0.15, # ~15% from MLS academies (Homegrown rate)
    'Euro from Tier 4': 0.005,        # ~0.5% reach top European leagues
}

scenarios = []
for tier in [1, 2, 3, 4]:
    invest = career_cost[tier]
    for scenario, schol_return, pro_return, p in [
        ('No outcome',           0,            0,       1.0),
        ('Partial schol (any)',  d2_men_4yr/2, 0,       prob.get('Any schol from Tier 3', 0.15) if tier==3 else 0.05),
        ('Full D1 schol',        d1_men_4yr,   0,       prob.get('D1 schol from Tier 3', 0.03) if tier==3 else 0.005),
        ('MLS career',           0,            mls_5yr, prob.get('Pro contract from Tier 3', 0.005) if tier < 4 else prob.get('Pro contract from Tier 4', 0.15)),
        ('European career',      0,            euro_5yr,prob.get('Euro from Tier 4', 0.001) if tier < 4 else prob.get('Euro from Tier 4', 0.005)),
    ]:
        net_return = schol_return + pro_return - invest
        ev_contribution = p * net_return
        scenarios.append({'tier': tier, 'scenario': scenario, 'investment': invest,
                           'return': schol_return+pro_return, 'net': net_return,
                           'probability': p, 'ev_contribution': ev_contribution})

df_scen = pd.DataFrame(scenarios)
ev_by_tier = df_scen.groupby('tier')['ev_contribution'].sum().reset_index()
ev_by_tier.columns = ['tier','expected_value_net_usd']
print('Expected net value (return minus cost) by tier:')
for _, row in ev_by_tier.iterrows():
    print(f'  Tier {int(row["tier"])}: ${row["expected_value_net_usd"]:,.0f}')

df_scen.to_csv('../data/analysis/roi_analysis.csv', index=False)

In [ ]:
# --- 5C: Monte Carlo simulation (Tier 3 player) ---
np.random.seed(42)
N = 100_000

# Draw random career costs (uniform between low and high)
cost_low  = costs[costs['tier']==3]['total_low'].mean()  * 7
cost_high = costs[costs['tier']==3]['total_high'].mean() * 7
sim_costs = np.random.uniform(cost_low, cost_high, N)

# Random outcome assignment
rand = np.random.random(N)
sim_returns = np.where(rand < 0.005,  mls_5yr,        # Pro (MLS)
              np.where(rand < 0.08,   d1_men_4yr,     # D1 scholarship
              np.where(rand < 0.35,   d2_men_4yr/2,   # Partial scholarship
                                      0)))             # No return

sim_net = sim_returns - sim_costs

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(sim_net, bins=80, color='#2980b9', alpha=0.75, edgecolor='white')
ax.axvline(sim_net.mean(), color='red', linestyle='--', label=f'Mean: ${sim_net.mean():,.0f}')
ax.axvline(0, color='black', linestyle='-', linewidth=1.5, label='Break-even')
ax.set_title('Monte Carlo: Net Financial Outcome for Tier 3 Player (100k simulations)', fontsize=12)
ax.set_xlabel('Net Return (USD)')
ax.set_ylabel('Frequency')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig('../data/analysis/monte_carlo_tier3.png', dpi=150)
plt.show()

print(f'\nMonte Carlo summary (Tier 3 player):')
print(f'  Mean net outcome:   ${sim_net.mean():,.0f}')
print(f'  Median net outcome: ${np.median(sim_net):,.0f}')
print(f'  % who break even:   {(sim_net >= 0).mean()*100:.1f}%')
print(f'  % who "go pro":     {(sim_returns >= mls_5yr).mean()*100:.2f}%')

In [ ]:
# --- 5D: Summary waterfall chart ---
fig, ax = plt.subplots(figsize=(10, 5))
tier_labels = ['Tier 1\nRec', 'Tier 2\nTravel', 'Tier 3\nECNL', 'Tier 4\nPro Academy']
ev_vals = ev_by_tier['expected_value_net_usd'].values
colors = ['#e74c3c' if v < 0 else '#27ae60' for v in ev_vals]
ax.bar(tier_labels, ev_vals, color=colors, alpha=0.85, edgecolor='white')
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Expected Net Financial Value of Youth Soccer Investment by Tier', fontsize=12)
ax.set_ylabel('Expected Net USD')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v:,.0f}'))
for i, v in enumerate(ev_vals):
    ax.text(i, v + (500 if v >= 0 else -1500), f'${v:,.0f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/analysis/roi_by_tier.png', dpi=150)
plt.show()